# Import Libraries

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"] = (20,10)

In [16]:
from sklearn.model_selection import train_test_split

In [17]:
from sklearn.preprocessing import PolynomialFeatures

In [18]:
from sklearn.linear_model import LinearRegression

In [19]:
from sklearn.metrics import mean_squared_error, r2_score

# Data Pre Processing

In [20]:
#Load Dataset

file_path = "/Users/saitarunsirapurapu/Downloads/Bengaluru_House_Data.csv"
df = pd.read_csv(file_path)

In [23]:
# data info
df.shape

(13320, 9)

In [25]:
df.groupby('area_type')['area_type'].agg('count')

area_type
Built-up  Area          2418
Carpet  Area              87
Plot  Area              2025
Super built-up  Area    8790
Name: area_type, dtype: int64

In [27]:
df1 = df.drop(['area_type','society','balcony','availability'],axis='columns')
df1.head(10)

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00
5,Whitefield,2 BHK,1170,2.0,38.00
6,Old Airport Road,4 BHK,2732,4.0,204.00
7,Rajaji Nagar,4 BHK,3300,4.0,600.00
8,Marathahalli,3 BHK,1310,3.0,63.25
9,Gandhi Bazar,6 Bedroom,1020,6.0,370.00


In [35]:
df1.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

In [36]:
df2 = df1.dropna()
df2.isnull().sum()

location      0
size          0
total_sqft    0
bath          0
price         0
dtype: int64

In [37]:
df2.shape

(13246, 5)

In [40]:
df2['size'].unique()

array(['2 BHK', '4 Bedroom', '3 BHK', '4 BHK', '6 Bedroom', '3 Bedroom',
       '1 BHK', '1 RK', '1 Bedroom', '8 Bedroom', '2 Bedroom',
       '7 Bedroom', '5 BHK', '7 BHK', '6 BHK', '5 Bedroom', '11 BHK',
       '9 BHK', '9 Bedroom', '27 BHK', '10 Bedroom', '11 Bedroom',
       '10 BHK', '19 BHK', '16 BHK', '43 Bedroom', '14 BHK', '8 BHK',
       '12 Bedroom', '13 BHK', '18 Bedroom'], dtype=object)

In [41]:
df2['bhk'] = df2['size'].apply(lambda x : int(x.split(' ')[0]))

/var/folders/b_/525vhd5s3v18vsy9rzxssxbh0000gn/T/ipykernel_96671/2499240759.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['bhk'] = df2['size'].apply(lambda x : int(x.split(' ')[0]))


In [42]:
df2.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00,4
2,Uttarahalli,3 BHK,1440,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00,3
4,Kothanur,2 BHK,1200,2.0,51.00,2


In [44]:
df2['bhk'].unique()

array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18])

In [46]:
df2.total_sqft.unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

In [47]:
def covert2float(x):
    try:
        float(x)
    except:
        return False
    return True

In [56]:
df2[~df2['total_sqft'].apply(covert2float)].head(10)

,location,size,total_sqft,bath,price,bhk
30,Yelahanka,4 BHK,2100 - 2850,4.0,186.000,4
122,Hebbal,4 BHK,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,2 BHK,1042 - 1105,2.0,54.005,2
165,Sarjapur,2 BHK,1145 - 1340,2.0,43.490,2
188,KR Puram,2 BHK,1015 - 1540,2.0,56.800,2
410,Kengeri,1 BHK,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,2 BHK,1195 - 1440,2.0,63.770,2
648,Arekere,9 Bedroom,4125Perch,9.0,265.000,9
661,Yelahanka,2 BHK,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,4 Bedroom,3090 - 5002,4.0,445.000,4


In [57]:
def convert2num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1]))/2
    try :
        return float(x)
    except :
        return None

In [58]:
convert2num('34.46 sqmeter') # test

In [59]:
df3 = df2.copy()
df3['total_sqft'] = df3['total_sqft'].apply(convert2num)

In [61]:
df3.head(5)

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3
4,Kothanur,2 BHK,1200.0,2.0,51.00,2


In [65]:
df3.loc[30] # .loc accessor is used for selecting data by label

location      Yelahanka
size              4 BHK
total_sqft       2475.0
bath                4.0
price             186.0
bhk                   4
Name: 30, dtype: object

# Feature transforming

In [68]:
df4 = df3.copy()
df4['price per sqft'] = df4['price']*100000 / df4['total_sqft'] # price is in lakhs Indian currency
df4.head()

,location,size,total_sqft,bath,price,bhk,price per sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2,4250.000000


In [69]:
df4.location.unique()

array(['Electronic City Phase II', 'Chikka Tirupathi', 'Uttarahalli', ...,
       '12th cross srinivas nagar banshankari 3rd stage',
       'Havanur extension', 'Abshot Layout'], dtype=object)

## how many rows are available per loaction in the datset

In [72]:
len(df4.location.unique()) # loaction is a categorical feature in here

1304

### 1304 is too many locations which is actually a categorical feature so this will lead to dimensionality curse 
### but we are going to use dimensionalitu reduction to resolve it.

### There are going to be loactions with one or two data points 

In [79]:
df4.location = df4.location.apply(lambda x : x.strip()) # to remove any leading/tailing spaces

location_stats = df4.groupby('location')['location'].agg('count').sort_values(ascending = True)

In [85]:
location_stats.head(1000)

location
1 Annasandrapalya       1
Kudlu Village,          1
Kumbhena Agrahara       1
Kuvempu Layout          1
LIC Colony              1
                       ..
Outer Ring Road East    8
Nelamangala             8
Vasanth nagar           8
Gopalkrishna Nagar      8
Kattigenahalli          8
Name: location, Length: 1000, dtype: int64

In [87]:
location_stats_less_than_5 = location_stats[location_stats<=5]
location_stats_less_than_5

location
1 Annasandrapalya      1
Kudlu Village,         1
Kumbhena Agrahara      1
Kuvempu Layout         1
LIC Colony             1
                      ..
Doddabanahalli         5
New Gurappana Palya    5
Kurubarahalli          5
Maithri Layout         5
9th Block Jayanagar    5
Name: location, Length: 908, dtype: int64

In [88]:
df4.location = df4.location.apply(lambda x: 'other' if x in location_stats_less_than_5 else x)
len(df4.location.unique())

386

# Eliminating Outliers 

In [91]:
# criteria for a proper bedroom

cpb = df4.total_sqft/df4.bhk<300

In [96]:
df4[cpb].head(100)

,location,size,total_sqft,bath,price,bhk,price per sqft
9,other,6 Bedroom,1020.0,6.0,370.0,6,36274.509804
45,HSR Layout,8 Bedroom,600.0,9.0,200.0,8,33333.333333
58,Murugeshpalya,6 Bedroom,1407.0,4.0,150.0,6,10660.980810
68,Devarachikkanahalli,8 Bedroom,1350.0,7.0,85.0,8,6296.296296
70,other,3 Bedroom,500.0,3.0,100.0,3,20000.000000
...,...,...,...,...,...,...,...
1802,other,4 Bedroom,712.0,2.0,67.5,4,9480.337079
1820,2nd Stage Nagarbhavi,5 Bedroom,1200.0,4.0,285.0,5,23750.000000
1825,Shivaji Nagar,7 Bedroom,1400.0,6.0,200.0,7,14285.714286
1830,Hosa Road,3 Bedroom,675.0,3.0,72.5,3,10740.740741


In [97]:
df4.shape

(13246, 7)

In [99]:
df5 = df4[~(cpb)]
df5.shape

(12502, 7)